## Feed-Forward Networks and AddNorm

Welcome back to Deconstructing the Transformer Architecture! In our second lesson, we're diving into the other essential components that make each Transformer block so powerful: the Position-wise Feed-Forward Network and the critical Add & Norm operations. While our previous lesson explored Multi-Head Attention and how multiple attention heads can capture diverse relationships across different representation subspaces, today we'll discover the complementary mechanisms that complete the Transformer's computational prowess.

While attention mechanisms handle the complex relationships between positions in a sequence, position-wise feed-forward networks provide the computational power to transform these attended representations. Think of attention as gathering the right information, and the feed-forward network as processing that information to extract meaningful patterns. Combined with residual connections and layer normalization, these components form the complete building blocks that enable Transformers to learn deep, stable representations. This lesson will guide you through implementing both the feed-forward networks and the Add & Norm components that make deep Transformer training possible.

---

### Understanding Position-wise Feed-Forward Networks

Position-wise Feed-Forward Networks serve a fundamentally different purpose than attention mechanisms in the Transformer architecture. While attention focuses on where to look across the sequence, the FFN determines what to do with the information once it's been gathered. The term "position-wise" indicates that the same transformation is applied independently to each position in the sequence, meaning the network processes each token's representation separately without considering relationships between positions.

The architecture is elegantly simple: two linear transformations with a non-linear activation function between them. Mathematically, this can be expressed as

$$\text{FFN}(x) = \max(0,\, xW_1 + b_1)\,W_2 + b_2$$

for ReLU activation, where the first linear layer typically expands the dimensionality (often to $4 \times d_{model}$), and the second layer projects back to the original model dimension. This expansion and contraction pattern allows the network to learn complex non-linear transformations while maintaining consistent dimensionality throughout the Transformer stack.

The key insight is that this component provides the Transformer with computational depth and non-linearity that pure attention mechanisms cannot achieve. Attention is fundamentally a weighted averaging operation, which is linear in nature. The position-wise FFN introduces the non-linear processing power that enables Transformers to learn complex function approximations and feature representations.

---

### Implementing the Position-wise Feed-Forward Network

Let's implement the core `PositionwiseFeedForward` network with configurable activation functions and proper weight initialization:

```python
class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1, activation='relu'):
        """
        Position-wise Feed-Forward Network

        Args:
            d_model: Model dimension
            d_ff: Feed-forward dimension (typically 4 * d_model)
            dropout: Dropout rate
            activation: Activation function ('relu' or 'gelu')
        """
        super(PositionwiseFeedForward, self).__init__()

        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

        # Choose activation function
        if activation.lower() == 'relu':
            self.activation = F.relu
        elif activation.lower() == 'gelu':
            self.activation = F.gelu
        else:
            raise ValueError(f"Unsupported activation: {activation}")

        self._init_weights()
```

The constructor establishes the essential FFN architecture with two linear transformations. The first layer expands from `d_model` to `d_ff` dimensions, providing the network with increased representational capacity in the intermediate layer. The second layer projects back to `d_model`, ensuring dimensional consistency with the rest of the Transformer. The dropout layer provides regularization to prevent overfitting, and the configurable activation function allows experimentation with different non-linearities.

The choice of `d_ff` is crucial: it's typically set to $4 \times d_{model}$ in standard Transformer implementations. This 4x expansion provides sufficient capacity for complex transformations while maintaining computational efficiency. The expansion-contraction pattern creates a bottleneck architecture that forces the network to learn compressed, meaningful representations in the intermediate layer.

---

### Activation Functions and Weight Initialization

The choice of activation function significantly impacts the FFN's behavior and training dynamics. Let's complete the implementation with proper weight initialization:

```python
def _init_weights(self):
    """Initialize weights"""
    nn.init.xavier_uniform_(self.linear1.weight)
    nn.init.xavier_uniform_(self.linear2.weight)
    nn.init.constant_(self.linear1.bias, 0)
    nn.init.constant_(self.linear2.bias, 0)

def forward(self, x):
    """Forward pass through position-wise FFN"""
    x = self.linear1(x)
    x = self.activation(x)
    x = self.dropout(x)
    x = self.linear2(x)
    return x
```

The forward pass implements the classic feed-forward pattern: linear transformation, activation, dropout, and final linear projection. Xavier uniform initialization ensures stable gradient flow during training by setting initial weights based on the number of input and output connections. The bias terms are initialized to zero, which is a common practice for linear layers in deep networks.

The activation function choice between ReLU and GELU represents different approaches to non-linearity. ReLU provides simple, efficient computation with sparse activation patterns, while GELU offers smoother gradients and has shown superior performance in many Transformer applications. GELU (Gaussian Error Linear Unit) approximates the activation pattern $x \cdot \Phi(x)$, where $\Phi(x)$ is the cumulative distribution function of the standard normal distribution, providing a more nuanced activation profile than the hard threshold of ReLU.

---

### Residual Connections and Layer Normalization

The Add & Norm component implements two critical techniques that enable stable training of deep Transformer networks: residual connections and layer normalization:

```python
class AddNorm(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        """
        Add & Norm layer: residual connection + layer normalization

        Args:
            d_model: Model dimension
            dropout: Dropout rate
        """
        super(AddNorm, self).__init__()
        self.layer_norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, sublayer_output):
        """
        Apply residual connection and layer normalization

        Args:
            x: Original input (residual connection)
            sublayer_output: Output from sublayer (attention or FFN)

        Returns:
            Normalized output after residual connection
        """
        # Add residual connection and apply dropout
        output = x + self.dropout(sublayer_output)

        # Apply layer normalization
        return self.layer_norm(output)
```

The `AddNorm` module encapsulates two fundamental techniques that solve different training challenges. Residual connections address the vanishing gradient problem by providing direct pathways for gradients to flow through the network. The mathematical operation $\text{output} = x + \text{sublayer}(x)$ ensures that even if the sublayer learns to output zeros, the original input can still pass through unchanged.

Layer normalization normalizes activations across the feature dimension for each individual example, stabilizing training dynamics and reducing internal covariate shift. Unlike batch normalization, layer normalization doesn't depend on batch statistics, making it more suitable for sequence processing where batch sizes might vary. The `eps` parameter is crucial for numerical stability, preventing division by zero when variance is small. While PyTorch's default is `1e-5`, using `1e-6` provides better stability in deep networks and mixed precision training. The learnable parameters gamma and beta are initialized to 1 and 0, allowing the network to adapt normalization strength per feature dimension.

The combination of these techniques enables the training of very deep Transformer networks with hundreds of layers.

---

### Combining Components in the Transformer Pattern

Now let's see how these components work together in the complete Transformer encoder layer pattern and examine the practical benefits:

```python
def test_transformer_block():
    """Test complete Transformer block pattern"""
    print("Testing Transformer Block Pattern...")

    batch_size, seq_len, d_model = 2, 8, 64
    num_heads, d_ff = 8, 256

    # Create components
    torch.manual_seed(42)
    x = torch.randn(batch_size, seq_len, d_model)

    mha = MultiHeadAttention(d_model, num_heads)
    ffn = PositionwiseFeedForward(d_model, d_ff)
    add_norm1 = AddNorm(d_model)
    add_norm2 = AddNorm(d_model)

    # Apply Transformer encoder layer pattern
    attn_output, _ = mha(x, x, x)
    x1 = add_norm1(x, attn_output)

    ffn_output = ffn(x1)
    x2 = add_norm2(x1, ffn_output)
```

This implementation demonstrates the canonical Transformer encoder layer pattern: Multi-Head Attention followed by Add & Norm, then Position-wise Feed-Forward Network followed by another Add & Norm. Each sublayer (attention and FFN) is wrapped with a residual connection and layer normalization, creating the characteristic two-step pattern that defines each Transformer block.

The sequential application shows how information flows through the block: first, the attention mechanism gathers relevant information from across the sequence; then, the Add & Norm operation stabilizes and normalizes this output while preserving the original input through the residual connection. Next, the FFN processes this normalized representation, and finally, another Add & Norm operation stabilizes the output while again preserving information flow through the residual connection.

---

### Output Discussion

When we run the complete test suite, we get the following output that demonstrates the effectiveness of our implementation:

```text
Testing Position-wise Feed-Forward Network...
Input shape: torch.Size([2, 8, 64])
FFN ReLU output shape: torch.Size([2, 8, 64])
FFN GELU output shape: torch.Size([2, 8, 64])
Testing Transformer Block Pattern...
Final output shape: torch.Size([2, 8, 64])
Input stats - mean: 0.0088, std: 1.0050
Output stats - mean: 0.0000, std: 1.0005
```

This output reveals several important characteristics of our implementation. First, the shape preservation confirms that both the FFN and the complete Transformer block maintain consistent dimensionality throughout processing. More importantly, the statistics comparison shows the effect of layer normalization: while the input has a small positive mean, the output is centered around zero, and the standard deviation remains close to 1.0, indicating that layer normalization is successfully stabilizing the activations and preventing activation drift that could destabilize training.

---

### Conclusion and Next Steps

We've successfully implemented the complete Position-wise Feed-Forward Network and Add & Norm components that form the other half of each Transformer block! These components work in harmony with the `MultiHeadAttention` mechanism you built previously to create powerful, trainable deep architectures. The FFN provides the non-linear computational power, while the Add & Norm operations ensure stable gradient flow and consistent activation magnitudes throughout the network.

The combination of these elements demonstrates the elegant engineering behind Transformers: attention mechanisms for relationship modeling, feed-forward networks for non-linear processing, and residual connections with layer normalization for training stability. In our next lesson, we'll explore how Transformers handle sequence order through positional encodings, completing our understanding of the core architectural components that make these models so effective across diverse NLP tasks. Now, let's get ready for some practice!

## Building Feed Forward Network Components

Now that you've learned about the key components of feed-forward networks, it's time to put that knowledge into practice by building the complete initialization logic.

---

### Your Task

In the provided code, you'll find the `PositionwiseFeedForward` class with an incomplete `__init__` method. Your job is to implement all the missing initialization components:

- Create the two linear transformation layers with the correct dimensions
- Set up the dropout layer with the specified rate
- Implement the activation function selection logic for both ReLU and GELU
- Add proper error handling for unsupported activation functions
- Call the weight initialization method

The `forward` method and weight initialization are already implemented to help you focus on understanding how all the pieces connect together. Once you complete this, you'll have a fully functional position-wise feed-forward network ready for the Transformer architecture!

---

### Starter Code

```python
import torch
import torch.nn as nn
import torch.nn.functional as F

class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1, activation='relu'):
        """
        Position-wise Feed-Forward Network
        
        Args:
            d_model: Model dimension
            d_ff: Feed-forward dimension (typically 4 * d_model)
            dropout: Dropout rate
            activation: Activation function ('relu' or 'gelu')
        """
        super(PositionwiseFeedForward, self).__init__()
        
        # TODO: Create the first linear layer (d_model -> d_ff)
        # TODO: Create the second linear layer (d_ff -> d_model)  
        # TODO: Create dropout layer with the specified dropout rate
        
        # TODO: Implement activation function selection logic
        # TODO: Handle 'relu' activation
        # TODO: Handle 'gelu' activation  
        # TODO: Raise ValueError for unsupported activation functions
        
        # TODO: Call the weight initialization method
    
    def _init_weights(self):
        """Initialize weights"""
        nn.init.xavier_uniform_(self.linear1.weight)
        nn.init.xavier_uniform_(self.linear2.weight)
        nn.init.constant_(self.linear1.bias, 0)
        nn.init.constant_(self.linear2.bias, 0)
    
    def forward(self, x):
        """Forward pass through position-wise FFN"""
        x = self.linear1(x)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.linear2(x)
        return x
```

---

### Solution

Only the TODOs in `__init__` were missing:

```python
self.linear1 = nn.Linear(d_model, d_ff)
self.linear2 = nn.Linear(d_ff, d_model)
self.dropout = nn.Dropout(dropout)

# Choose activation function
if activation.lower() == 'relu':
    self.activation = F.relu
elif activation.lower() == 'gelu':
    self.activation = F.gelu
else:
    raise ValueError(f"Unsupported activation: {activation}")

self._init_weights()
```

- `nn.Linear(d_model, d_ff)` expands each position's vector from `d_model` to `d_ff` dimensions, and `nn.Linear(d_ff, d_model)` projects it back. Because both layers act on the last dimension only, every position is transformed independently, which is what "position-wise" means.
- `nn.Dropout(dropout)` stores the dropout layer that `forward` applies between the two linear layers.
- The activation is stored as a function (`F.relu` or `F.gelu`) so `forward` can call `self.activation(x)` without caring which one was chosen. `activation.lower()` makes the check case-insensitive, so `'ReLU'` and `'GELU'` also work.
- Any other value reaches the `else` branch and raises a `ValueError` with a clear message, instead of failing later with a confusing error.
- `self._init_weights()` must be called last, after `linear1` and `linear2` exist, because it uses `self.linear1.weight` and `self.linear2.weight`.
- Without the fix, `__init__` creates nothing. The first call to `forward` fails with an `AttributeError`, because `self.linear1`, `self.dropout` and `self.activation` do not exist.

This exercise has no `main()`, so the cell below just defines the full `PositionwiseFeedForward` class with the fix applied. For an input `x` of shape `(batch_size, seq_len, d_model)`, the output has the same shape.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1, activation='relu'):
        """
        Position-wise Feed-Forward Network
        
        Args:
            d_model: Model dimension
            d_ff: Feed-forward dimension (typically 4 * d_model)
            dropout: Dropout rate
            activation: Activation function ('relu' or 'gelu')
        """
        super(PositionwiseFeedForward, self).__init__()
        
        # Two linear layers: expand d_model -> d_ff, then project back d_ff -> d_model
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        
        # Choose activation function
        if activation.lower() == 'relu':
            self.activation = F.relu
        elif activation.lower() == 'gelu':
            self.activation = F.gelu
        else:
            raise ValueError(f"Unsupported activation: {activation}")
        
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights"""
        nn.init.xavier_uniform_(self.linear1.weight)
        nn.init.xavier_uniform_(self.linear2.weight)
        nn.init.constant_(self.linear1.bias, 0)
        nn.init.constant_(self.linear2.bias, 0)
    
    def forward(self, x):
        """Forward pass through position-wise FFN"""
        x = self.linear1(x)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.linear2(x)
        return x

## Initialize Network Weights

Now that you've successfully set up the basic structure of the `PositionwiseFeedForward` class, it's time to focus on proper weight initialization to ensure that your network trains effectively.

---

### Your Task

Your goal is to complete the `_init_weights` method in the provided code. Currently, this method contains only placeholder comments and a `pass` statement, but proper weight initialization is crucial for stable training in deep networks.

You'll need to apply Xavier uniform initialization to both linear layer weights, and also set all bias parameters to zero. Once you complete this step, your FFN will be ready for robust training!

---

### Starter Code

```python
import torch
import torch.nn as nn
import torch.nn.functional as F

class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1, activation='relu'):
        """
        Position-wise Feed-Forward Network
        
        Args:
            d_model: Model dimension
            d_ff: Feed-forward dimension (typically 4 * d_model)
            dropout: Dropout rate
            activation: Activation function ('relu' or 'gelu')
        """
        super(PositionwiseFeedForward, self).__init__()
        
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        
        # Choose activation function
        if activation.lower() == 'relu':
            self.activation = F.relu
        elif activation.lower() == 'gelu':
            self.activation = F.gelu
        else:
            raise ValueError(f"Unsupported activation: {activation}")
        
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights"""
        # TODO: Initialize self.linear1.weight using Xavier uniform initialization
        # TODO: Initialize self.linear2.weight using Xavier uniform initialization
        # TODO: Initialize self.linear1.bias to 0
        # TODO: Initialize self.linear2.bias to 0
        pass 
    
    def forward(self, x):
        """Forward pass through position-wise FFN"""
        x = self.linear1(x)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.linear2(x)
        return x
```

---

### Solution

Only the body of `_init_weights` was missing. The four TODO comments and the `pass` statement become:

```python
nn.init.xavier_uniform_(self.linear1.weight)
nn.init.xavier_uniform_(self.linear2.weight)
nn.init.constant_(self.linear1.bias, 0)
nn.init.constant_(self.linear2.bias, 0)
```

- `nn.init.xavier_uniform_(...)` fills each weight matrix with values drawn from a uniform distribution whose range depends on the layer's input and output sizes (its fan-in and fan-out). This keeps the scale of activations and gradients roughly stable from layer to layer, which helps training start smoothly.
- `nn.init.constant_(..., 0)` sets every bias to zero, so the layers start as pure linear maps and the biases are learned during training.
- Both layers need the same treatment, one line per weight and one per bias, which gives the four lines above. Unlike the multi-head attention version of this method, there is no loop and no `bias is not None` check, because `linear1` and `linear2` are created with the default `bias=True`.
- The trailing underscore in `xavier_uniform_` and `constant_` marks in-place operations: they modify the parameters directly and return nothing you need to assign.
- Without the fix, the method body is just `pass`. The code still runs, but the layers keep PyTorch's default `nn.Linear` initialization instead of Xavier uniform weights and zero biases, so the mistake is easy to miss.

This exercise has no `main()`, so the cell below just defines the full `PositionwiseFeedForward` class with the fix applied. After constructing it, for example `PositionwiseFeedForward(64, 256)`, both `linear1.bias` and `linear2.bias` are all zeros.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1, activation='relu'):
        """
        Position-wise Feed-Forward Network
        
        Args:
            d_model: Model dimension
            d_ff: Feed-forward dimension (typically 4 * d_model)
            dropout: Dropout rate
            activation: Activation function ('relu' or 'gelu')
        """
        super(PositionwiseFeedForward, self).__init__()
        
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        
        # Choose activation function
        if activation.lower() == 'relu':
            self.activation = F.relu
        elif activation.lower() == 'gelu':
            self.activation = F.gelu
        else:
            raise ValueError(f"Unsupported activation: {activation}")
        
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights"""
        # Xavier uniform for both weight matrices
        nn.init.xavier_uniform_(self.linear1.weight)
        nn.init.xavier_uniform_(self.linear2.weight)
        # Zero biases
        nn.init.constant_(self.linear1.bias, 0)
        nn.init.constant_(self.linear2.bias, 0)
    
    def forward(self, x):
        """Forward pass through position-wise FFN"""
        x = self.linear1(x)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.linear2(x)
        return x

## Building Transformer Stability Components

After working with feed-forward networks, you're ready to tackle another key Transformer component: the `AddNorm` layer. This powerful combination of residual connections and layer normalization helps deep networks train without gradient problems.

---

### Your Task

Your goal is to complete the `forward` method in the `AddNorm` class. The provided code gives you the framework, but the core implementation is missing. You need to:

- Apply dropout to the sublayer output
- Add the residual connection using the original input
- Apply layer normalization to the result

Pay attention to the order of these operations — getting the sequence right is crucial for proper functionality. Once you complete this, you'll have mastered a key building block for stable Transformer training!

---

### Starter Code

```python
import torch
import torch.nn as nn

class AddNorm(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        """
        Add & Norm layer: residual connection + layer normalization
        
        Args:
            d_model: Model dimension
            dropout: Dropout rate
        """
        super(AddNorm, self).__init__()
        self.layer_norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, sublayer_output):
        """
        Apply residual connection and layer normalization
        
        Args:
            x: Original input (residual connection)
            sublayer_output: Output from sublayer (attention or FFN)
        
        Returns:
            Normalized output after residual connection
        """
        # TODO: Apply dropout to the sublayer_output

        # TODO: Add the original input x (residual connection) 

        # TODO: Apply layer normalization to the result
        pass
```

---

### Solution

Only the body of `forward` was missing. The three TODO comments and the `pass` statement become:

```python
# Apply dropout to the sublayer output, then add the residual connection
output = x + self.dropout(sublayer_output)

# Apply layer normalization to the result
return self.layer_norm(output)
```

- `self.dropout(sublayer_output)` randomly zeroes parts of the sublayer output during training. It is applied to the sublayer output only, not to `x`, so the original input passes through the residual path untouched.
- `x + ...` adds the residual connection. Even if the sublayer output is close to zero, `x` still flows through unchanged, which gives gradients a direct path through the network. In math terms: $\text{output} = x + \text{Dropout}(\text{sublayer}(x))$.
- `self.layer_norm(output)` normalizes each position's vector across the feature dimension. It comes last, so it normalizes the sum of the residual and the sublayer output. This is the post-norm ordering of the original Transformer.
- The order matters: dropout, then add, then normalize. Normalizing before the add, or applying dropout after the add, would give a different (and incorrect) layer.
- Without the fix, the method body is just `pass`, so `forward` returns `None`. Any code that uses the result, such as passing it to the next sublayer, fails.

This exercise has no `main()`, so the cell below just defines the full `AddNorm` class with the fix applied. For `x` and `sublayer_output` of shape `(batch_size, seq_len, d_model)`, the output has the same shape.

In [ ]:
import torch
import torch.nn as nn

class AddNorm(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        """
        Add & Norm layer: residual connection + layer normalization
        
        Args:
            d_model: Model dimension
            dropout: Dropout rate
        """
        super(AddNorm, self).__init__()
        self.layer_norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, sublayer_output):
        """
        Apply residual connection and layer normalization
        
        Args:
            x: Original input (residual connection)
            sublayer_output: Output from sublayer (attention or FFN)
        
        Returns:
            Normalized output after residual connection
        """
        # Dropout on the sublayer output, then add the residual connection
        output = x + self.dropout(sublayer_output)

        # Layer normalization on the sum
        return self.layer_norm(output)

## Building Your First Transformer Block

Now that you've mastered the `AddNorm` forward pass, it's time to see how all the Transformer components work together in practice. Your goal is to complete the data flow through a complete Transformer encoder sub-block by connecting the multi-head attention, feed-forward network, and Add & Norm components in the correct sequence.

---

### Your Task

In the provided code, you'll find all the necessary components already instantiated in the `test_transformer_block` function. The attention output has been computed, but the remaining connections are missing. You need to:

- Apply the first Add & Norm using the original input and the attention output
- Feed the result through the position-wise feed-forward network
- Apply the second Add & Norm using the appropriate inputs

Pay close attention to which outputs become inputs for the next step — this is the heart of how Transformer blocks process information through multiple layers while maintaining stable gradients.

---

### Starter Code

```python
import torch
import torch.nn as nn

from transformer.feed_forward import PositionwiseFeedForward
from transformer.utils import AddNorm
from transformer.attention import MultiHeadAttention

def test_feed_forward():
    """Test Position-wise Feed-Forward Network"""
    print("Testing Position-wise Feed-Forward Network...")
    
    batch_size, seq_len, d_model = 2, 8, 64
    d_ff = 256  # Typically 4 * d_model
    
    # Create sample input
    torch.manual_seed(42)
    x = torch.randn(batch_size, seq_len, d_model)
    
    # Test with different activations
    ffn_relu = PositionwiseFeedForward(d_model, d_ff, activation='relu')
    ffn_gelu = PositionwiseFeedForward(d_model, d_ff, activation='gelu')
    
    output_relu = ffn_relu(x)
    output_gelu = ffn_gelu(x)
    
    print(f"Input shape: {x.shape}")
    print(f"FFN ReLU output shape: {output_relu.shape}")
    print(f"FFN GELU output shape: {output_gelu.shape}")
    
    # Verify dimensions preserved
    assert output_relu.shape == x.shape
    assert output_gelu.shape == x.shape
    
    return ffn_relu

def test_transformer_block():
    """Test complete Transformer block pattern"""
    print("Testing Transformer Block Pattern...")
    
    batch_size, seq_len, d_model = 2, 8, 64
    num_heads, d_ff = 8, 256
    
    # Create components
    torch.manual_seed(42)
    x = torch.randn(batch_size, seq_len, d_model)
    
    mha = MultiHeadAttention(d_model, num_heads)
    ffn = PositionwiseFeedForward(d_model, d_ff)
    add_norm1 = AddNorm(d_model)
    add_norm2 = AddNorm(d_model)
    
    # Apply Transformer encoder layer pattern
    attn_output, _ = mha(x, x, x)
    
    # TODO: Apply the first Add & Norm. 
    #       The input to this AddNorm is the original x and the attention output.
    
    # TODO: Apply the Position-wise Feed-Forward Network.
    #       The input is the output from the first Add & Norm.
    
    # TODO: Apply the second Add & Norm.
    #       The input to this AddNorm is the output from the first Add & Norm (x1)
    #       and the FFN output.
    
    print(f"Final output shape: {x2.shape}")
    print(f"Input stats - mean: {x.mean():.4f}, std: {x.std():.4f}")
    print(f"Output stats - mean: {x2.mean():.4f}, std: {x2.std():.4f}")
    
    return x2

def main():
    ffn = test_feed_forward()
    final_output = test_transformer_block()
    

if __name__ == "__main__":
    main()
```

---

### Solution

Only the three steps in `test_transformer_block` were missing. The TODO comments become:

```python
# First Add & Norm: original input x + attention output
x1 = add_norm1(x, attn_output)

# Position-wise feed-forward network on the normalized result
ffn_output = ffn(x1)

# Second Add & Norm: x1 + FFN output
x2 = add_norm2(x1, ffn_output)
```

- `add_norm1(x, attn_output)` wraps the attention sublayer. The first argument is the residual input (`x`) and the second is the sublayer output (`attn_output`), matching the `AddNorm.forward(x, sublayer_output)` signature. The result is named `x1` because the later TODO comments refer to it.
- `ffn(x1)` feeds the output of the first Add & Norm, not the original `x`, into the feed-forward network. Each sublayer works on the result of the previous one.
- `add_norm2(x1, ffn_output)` wraps the FFN sublayer. The residual input is now `x1`, the input to the FFN, and not `x`.
- The result must be called `x2`, because the print statements after the TODOs use `x2.shape`, `x2.mean()` and `x2.std()`.
- This is the encoder pattern from the lesson: attention, Add & Norm, feed-forward, Add & Norm.
- Without the fix, `x2` is never defined, so `test_transformer_block` fails with a `NameError` at the first `print` that uses it.

The starter code imports `PositionwiseFeedForward`, `AddNorm` and `MultiHeadAttention` from CodeSignal's `transformer` package, which does not exist in this notebook. The cell below defines all three classes inline, using the versions built in this and the previous lesson, and keeps the `main()` from the starter code. Running it should print the shape and statistics lines shown in the lesson's "Output Discussion" section.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# The `transformer` package is CodeSignal-only, so the three components are defined inline.

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        """
        Multi-Head Attention module
        
        Args:
            d_model: Model dimension
            num_heads: Number of attention heads
            dropout: Dropout rate
        """
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Linear projections for Q, K, V and output
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout)
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights using Xavier uniform initialization"""
        for linear in [self.w_q, self.w_k, self.w_v, self.w_o]:
            nn.init.xavier_uniform_(linear.weight)
            if linear.bias is not None:
                nn.init.constant_(linear.bias, 0)
    
    def scaled_dot_product_attention(self, q, k, v, mask=None):
        """Scaled dot-product attention for multiple heads"""
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        
        output = torch.matmul(attention_weights, v)
        return output, attention_weights
    
    def forward(self, query, key, value, mask=None):
        """Forward pass for multi-head attention"""
        batch_size, seq_len_q = query.size(0), query.size(1)
        seq_len_k = key.size(1)
        
        # Linear projections and reshape for multi-head attention
        Q = self.w_q(query).view(batch_size, seq_len_q, self.num_heads, self.d_k).transpose(1, 2)
        K = self.w_k(key).view(batch_size, seq_len_k, self.num_heads, self.d_k).transpose(1, 2)
        V = self.w_v(value).view(batch_size, seq_len_k, self.num_heads, self.d_k).transpose(1, 2)
        
        # Adjust mask for multi-head attention
        if mask is not None:
            if mask.dim() == 2:  # [seq_len, seq_len]
                mask = mask.unsqueeze(0).unsqueeze(0)  # [1, 1, seq_len, seq_len]
            elif mask.dim() == 3:  # [batch_size, seq_len, seq_len]
                mask = mask.unsqueeze(1)  # [batch_size, 1, seq_len, seq_len]
        
        # Apply attention
        attention_output, attention_weights = self.scaled_dot_product_attention(Q, K, V, mask)
        
        # Concatenate heads and apply output projection
        attention_output = attention_output.transpose(1, 2).contiguous()
        attention_output = attention_output.view(batch_size, seq_len_q, self.d_model)
        output = self.w_o(attention_output)
        
        return output, attention_weights


class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1, activation='relu'):
        """
        Position-wise Feed-Forward Network
        
        Args:
            d_model: Model dimension
            d_ff: Feed-forward dimension (typically 4 * d_model)
            dropout: Dropout rate
            activation: Activation function ('relu' or 'gelu')
        """
        super(PositionwiseFeedForward, self).__init__()
        
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        
        # Choose activation function
        if activation.lower() == 'relu':
            self.activation = F.relu
        elif activation.lower() == 'gelu':
            self.activation = F.gelu
        else:
            raise ValueError(f"Unsupported activation: {activation}")
        
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights"""
        nn.init.xavier_uniform_(self.linear1.weight)
        nn.init.xavier_uniform_(self.linear2.weight)
        nn.init.constant_(self.linear1.bias, 0)
        nn.init.constant_(self.linear2.bias, 0)
    
    def forward(self, x):
        """Forward pass through position-wise FFN"""
        x = self.linear1(x)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.linear2(x)
        return x


class AddNorm(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        """
        Add & Norm layer: residual connection + layer normalization
        
        Args:
            d_model: Model dimension
            dropout: Dropout rate
        """
        super(AddNorm, self).__init__()
        self.layer_norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, sublayer_output):
        """
        Apply residual connection and layer normalization
        
        Args:
            x: Original input (residual connection)
            sublayer_output: Output from sublayer (attention or FFN)
        
        Returns:
            Normalized output after residual connection
        """
        output = x + self.dropout(sublayer_output)
        return self.layer_norm(output)


def test_feed_forward():
    """Test Position-wise Feed-Forward Network"""
    print("Testing Position-wise Feed-Forward Network...")
    
    batch_size, seq_len, d_model = 2, 8, 64
    d_ff = 256  # Typically 4 * d_model
    
    # Create sample input
    torch.manual_seed(42)
    x = torch.randn(batch_size, seq_len, d_model)
    
    # Test with different activations
    ffn_relu = PositionwiseFeedForward(d_model, d_ff, activation='relu')
    ffn_gelu = PositionwiseFeedForward(d_model, d_ff, activation='gelu')
    
    output_relu = ffn_relu(x)
    output_gelu = ffn_gelu(x)
    
    print(f"Input shape: {x.shape}")
    print(f"FFN ReLU output shape: {output_relu.shape}")
    print(f"FFN GELU output shape: {output_gelu.shape}")
    
    # Verify dimensions preserved
    assert output_relu.shape == x.shape
    assert output_gelu.shape == x.shape
    
    return ffn_relu

def test_transformer_block():
    """Test complete Transformer block pattern"""
    print("Testing Transformer Block Pattern...")
    
    batch_size, seq_len, d_model = 2, 8, 64
    num_heads, d_ff = 8, 256
    
    # Create components
    torch.manual_seed(42)
    x = torch.randn(batch_size, seq_len, d_model)
    
    mha = MultiHeadAttention(d_model, num_heads)
    ffn = PositionwiseFeedForward(d_model, d_ff)
    add_norm1 = AddNorm(d_model)
    add_norm2 = AddNorm(d_model)
    
    # Apply Transformer encoder layer pattern
    attn_output, _ = mha(x, x, x)
    
    # First Add & Norm: original input x + attention output
    x1 = add_norm1(x, attn_output)
    
    # Position-wise FFN on the output of the first Add & Norm
    ffn_output = ffn(x1)
    
    # Second Add & Norm: x1 + FFN output
    x2 = add_norm2(x1, ffn_output)
    
    print(f"Final output shape: {x2.shape}")
    print(f"Input stats - mean: {x.mean():.4f}, std: {x.std():.4f}")
    print(f"Output stats - mean: {x2.mean():.4f}, std: {x2.std():.4f}")
    
    return x2

def main():
    ffn = test_feed_forward()
    final_output = test_transformer_block()
    

if __name__ == "__main__":
    main()